# Gestion y Analisis de Archivos de Texto

Ejercicios para leer, procesar y extraer informacion de archivos de texto. Habilidades fundamentales en cualquier pipeline de PLN.

## Ejercicio 1: Lectura y estadisticas basicas

1. Crea un archivo de texto con varios parrafos.
2. Lee el contenido y calcula estadisticas: palabras, oraciones, parrafos, caracteres.
3. Imprime un reporte resumido.

In [ ]:
import re

contenido = """
El procesamiento de lenguaje natural estudia como las computadoras interactuan con el lenguaje humano.
Es una rama de la inteligencia artificial con aplicaciones en traductores, chatbots y buscadores.

La tokenizacion divide el texto en unidades minimas llamadas tokens.
Cada token puede ser una palabra, un simbolo o un numero.
Este proceso es el primer paso en casi todo pipeline de PLN.

Los modelos de lenguaje aprenden patrones estadisticos del texto.
Con suficientes datos pueden predecir palabras, clasificar documentos y responder preguntas.
Los transformers son actualmente la arquitectura mas utilizada.
"""

with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write(contenido)

with open("corpus.txt", "r", encoding="utf-8") as f:
    texto = f.read()

parrafos  = [p.strip() for p in texto.split("\n\n") if p.strip()]
oraciones = re.split(r'[.!?]+', texto)
oraciones = [o.strip() for o in oraciones if o.strip()]
palabras  = texto.split()

print("=== Reporte del archivo corpus.txt ===")
print(f"Caracteres  : {len(texto)}")
print(f"Palabras    : {len(palabras)}")
print(f"Oraciones   : {len(oraciones)}")
print(f"Parrafos    : {len(parrafos)}")
print(f"Promedio palabras/oracion: {len(palabras)/len(oraciones):.1f}")
print("\nParrafos encontrados:")
for i, p in enumerate(parrafos, 1):
    print(f"  [{i}] {p[:60]}...")


## Ejercicio 2: Extraccion de informacion

1. Cuenta la frecuencia de cada palabra.
2. Encuentra las palabras mas largas.
3. Detecta palabras repetidas en oraciones.

In [ ]:
from collections import Counter
import re

def limpiar(t):
    return re.sub(r'[^a-z\s]', '', t.lower()).split()

tokens = limpiar(texto)
frecuencia = Counter(tokens)

print("Top 10 palabras mas frecuentes:")
for palabra, n in frecuencia.most_common(10):
    barra = '#' * n
    print(f"  {palabra:<20} {barra} ({n})")

print("\nPalabras mas largas (top 5):")
por_longitud = sorted(set(tokens), key=len, reverse=True)
for p in por_longitud[:5]:
    print(f"  '{p}' ({len(p)} letras)")

print("\nPalabras que aparecen una sola vez (hapax legomena):")
hapax = [p for p, n in frecuencia.items() if n == 1]
print(f"  Total: {len(hapax)}")
print(f"  Ejemplos: {hapax[:8]}")


## Ejercicio 3: Resumen extractivo basico

1. Puntua cada oracion segun cuantas palabras importantes contiene.
2. Selecciona las N oraciones con mayor puntaje.
3. Las oraciones seleccionadas forman el resumen.

In [ ]:
from collections import Counter
import re

def puntuar_oraciones(texto, top_n=3):
    """Resumen extractivo basado en frecuencia de palabras."""
    palabras_vacias = {
        'el','la','los','las','un','una','de','del','en','es','con',
        'se','que','por','para','y','a','al','su','sus','lo','le'
    }

    oraciones = re.split(r'(?<=[.!?])\s+', texto.strip())
    oraciones = [o for o in oraciones if len(o.split()) > 4]

    # Frecuencia de palabras significativas
    tokens = re.sub(r'[^a-z\s]', '', texto.lower()).split()
    tokens_sig = [t for t in tokens if t not in palabras_vacias and len(t) > 3]
    freq = Counter(tokens_sig)

    # Puntaje: suma de frecuencias de palabras en la oracion
    puntajes = []
    for oracion in oraciones:
        words = re.sub(r'[^a-z\s]', '', oracion.lower()).split()
        score = sum(freq.get(w, 0) for w in words if w not in palabras_vacias)
        puntajes.append((score, oracion))

    # Ordenar y seleccionar top_n
    puntajes.sort(reverse=True)
    resumen = [o for _, o in puntajes[:top_n]]

    return resumen, puntajes

resumen, puntajes = puntuar_oraciones(texto, top_n=3)

print("=== Puntajes por oracion ===")
for score, oracion in sorted(puntajes, reverse=True):
    print(f"  [{score:2d}] {oracion[:65]}...")

print("\n=== Resumen extractivo (top 3 oraciones) ===")
for i, oracion in enumerate(resumen, 1):
    print(f"  {i}. {oracion}")
